# 07 — Simulation and Scenario Testing

This notebook shifts the project from prediction into simulation-driven exploration. In this next iteration, I used simulation to uncover what my models actually believe, and where they break.

Rather than focusing primarily on causal identification, the goal is to use Monte Carlo simulation and structured scenario testing to:

- stress test model behavior under alternative conditions
- evaluate sensitivity to key inputs like starting position and reliability
- uncover where models are stable vs fragile
- identify missing structure in the dataset
- generate ideas for new features, transformations, and modeling approaches

In this sense, simulation is not just a forecasting tool—it is a diagnostic tool.

It provides a way to ask:

**What does the model believe about the world, and where does that belief break down?**

This notebook uses Formula 1 as a controlled environment to explore:
- how race and season outcomes respond to perturbations
- how uncertainty accumulates across a season
- and how model assumptions translate into simulated reality

These insights are not limited to motorsport. They generalize to any system where:
- outcomes are stochastic
- structure is partially observed
- and models must operate under uncertainty

## Predictive Simulation vs Counterfactual Reasoning

This notebook makes use of both predictive simulation and counterfactual scenario testing.

- **Predictive simulation** draws outcomes from the model’s learned distribution
- **Counterfactual scenarios** alter inputs and observe how simulated outcomes change

These exercises can sometimes resemble causal analysis, but they are not equivalent.

A fully causal interpretation would require stronger identification assumptions than are available in this dataset.

Instead, the focus here is:

- understanding model sensitivity
- exploring alternative racing scenarios
- identifying where model assumptions hold or fail

The goal is not to claim definitive causal effects, but to use simulation as a structured way to probe the model and the data.

## Why Simulation Matters in This Project

Up to this point, the project has focused on:
- feature engineering
- model comparison
- validation and stability

Those steps answer an important question:

**How well do the models fit the data?**

Simulation allows us to ask a different question:

**What do these models imply about how the system behaves?**

By simulating alternative scenarios, we can:
- observe how small changes propagate through the system
- identify nonlinear effects that are not obvious from model coefficients
- detect where predictions become unstable
- uncover patterns that suggest missing variables or transformations

In other words, simulation turns static model evaluation into dynamic system exploration.

## Section 1: Data Inputs for Simulation and Setup

- race-level expected finish or probability estimates
- points expectations
- driver / constructor identifiers
- grid / qualifying info
- reliability proxies
- weather flags
- pit-stop shock parameters

-----------------------------------------------------

- predicted finish position
- predicted points
- podium probability
- points probability
- DNF probability if available from prior work

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone

## Load Simulation Base Table

Use the cleaned pre-race feature store and model outputs from earlier notebooks as inputs into the simulation framework. What distribution are we simulating from?

In [ ]:
SIM_FEATURES = [
    "year",
    "round",
    "raceId",
    "driverId",
    "constructorId",
    "qualifying_position",
    "grid_clean",
    "driver_avg_finish_last5",
    "driver_dnf_rate_last5",
    "constructor_avg_finish_last5",
    "constructor_dnf_rate_last5",
    "is_wet_race",
    "alt",
    "temp_avg",
    "precipitation",
    "wind_speed"
]

## Section 2 Monte Carlo Simulation Design

**Monte Carlo Simulation: Season Outcomes Under Reliability Assumptions**

This section simulates season standings under different reliability assumptions by altering DNF probabilities and re-running race outcomes thousands of times.

Scenarios:
- baseline
- +10% reliability improvement for constructor
- -10% reliability
- selective reliability shock to one driver

Outputs:
- championship probability
- average constructor points
- standing distributions

If reliability improves or worsens for a team, how do season standings change?

For each race-driver:

**Monte Carlo: Reliability as a Structural Stress Test**

This simulation examines how sensitive model outcomes are to reliability assumptions.

Rather than treating reliability purely as a causal driver, this section uses it as a stress test:

- How much do rare events (DNFs) influence season outcomes?
- Do models understate or overstate the impact of reliability?
- How does uncertainty compound across multiple races?

This helps reveal:
- whether the model is overly dependent on clean finishes
- whether tail risks are properly captured
- whether reliability should be modeled more explicitly

possible insights:
- maybe you need better DNF modeling
- maybe binary outcomes need separate treatment
- maybe survival / hazard modeling could be added later

## Counterfactual Grid Positions: Sensitivity to Starting Conditions

This section tests how sensitive model outcomes are to changes in starting position.

The question we want to answer is:

how are podium and points probabilities changed when starting position is shifted upward or downward, while holding other pre-race features fixed?

Rather than interpreting this strictly as a causal estimate, the goal is to understand:

- how much the model relies on grid position
- whether that relationship is linear or nonlinear
- where diminishing returns begin

By shifting starting positions and re-simulating outcomes, we can observe:
- marginal gains in expected points
- nonlinear effects near the front of the grid
- differences across drivers and constructors

This provides insight into:
- whether grid position is over-weighted or under-weighted
- whether transformations (e.g., nonlinear terms, splines) may improve the model

## Race-Day Shocks: Exploring Model Blind Spots

This section introduces stylized shocks—weather variation and pit-stop disruptions—to explore how models respond to race-day volatility.

The goal is not to simulate exact race conditions, but to test:

- how robust predictions are to unexpected events
- whether the model captures variability in chaotic conditions
- where predictions break down

If simulated outcomes change dramatically under modest shocks, that suggests:
- missing variables
- insufficient modeling of interactions
- or underestimation of variance

This section is particularly useful for identifying:
- gaps in the feature set
- potential interaction effects (e.g., weather × strategy)
- opportunities for richer modeling approaches

## What the Simulations Reveal

The most valuable outcome of this notebook is not the simulated standings themselves, but what those simulations reveal about the models.

Across scenarios, several patterns emerge:

- Some variables have stable, predictable effects across simulations
- Others introduce large variability depending on context
- Certain outcomes are highly sensitive to small input changes
- Some sources of variation are not well captured by the current feature set

These findings suggest that:

- not all noise is removable
- some “noise” reflects unobserved structure
- model performance depends on how well that structure is approximated

More importantly, the simulations point toward next steps:
- refining reliability modeling
- introducing nonlinear transformations
- exploring interaction effects
- improving uncertainty modeling

In this way, simulation becomes a tool for **model development**, not just model evaluation.

## Instrumental Variables: A Possible Future Direction

A fully causal interpretation of counterfactual starting positions would require stronger identification than is currently available in this dataset.

One possible future direction is to explore instrumental variables based on exogenous grid shocks, such as penalties or irregular qualifying disruptions. However, a credible instrument would need to affect starting position without directly affecting race outcomes through other channels.

At this stage, the notebook focuses on predictive counterfactuals rather than fully identified causal estimates.

Potentially:

- weather shock affecting qualifying or starting position but not race outcome except through starting position
** hard to defend because weather likely affects race outcomes directly too.

Another possible instrument:

- qualifying penalties / grid penalties
** may correlate with team reliability or strategy, so also not clean.

## Must-have visuals
1. Simulated constructor standings distribution
violin / box plots by scenario
2. Championship probability bars
baseline vs reliability scenarios
3. Points / podium probability change under grid shifts
line plot of probability vs starting position change
4. Season simulation fan chart
simulated points trajectories
5. Scenario comparison heatmap
constructors × scenarios
6. Counterfactual uplift plot
expected points gain from better starting positions

## Overarching Question

**How much of a season is shaped by what should have happened, and how much changes when we intervene on starting position, reliability, or race-day shocks?**

## Connecting Back to the Broader Project

This notebook builds on earlier work in the project:

- distribution analysis showed that the data is not well-behaved
- model comparison revealed tradeoffs between interpretability and performance
- resampling highlighted instability and sensitivity to data splits

Simulation brings these ideas together by showing:

- how uncertainty propagates through the system
- how model assumptions behave under stress
- and where the data fails to fully describe reality

This creates a natural bridge to the next phase of the project, which will focus on:

- probabilistic modeling
- scenario optimization
- and decision-oriented analysis